# Customer Purchase Behavior Analyzer

**Objective:** Load customer, sales and product data from 3 different formats (CSV, JSON, SQL), clean it, engineer new features and prepare one final dataset for analysis / ML.


## 1. Data Understanding & Loading

We have 3 files:
- `users.csv` - customer details
- `sales.json` - transaction details
- `inventory.sql` - product details (SQL file, we will run it in an in-memory SQLite database)

In [1]:
import pandas as pd
import numpy as np
import json
import sqlite3
import matplotlib.pyplot as plt

# so numbers don't print in scientific notation, easier to read
pd.set_option('display.float_format', lambda x: '%.2f' % x)


In [2]:
# 1.1 Load users.csv
users_df = pd.read_csv('data/users.csv')
print("Users shape:", users_df.shape)
users_df.head()


Users shape: (200, 6)


,user_id,name,age,gender,city,registration_date
0,U0001,Vihaan Sharma,35,Other,Jaipur,2022-09-08
1,U0002,Sai Reddy,30,Other,Hyderabad,2023-11-24
2,U0003,Aarohi Gupta,37,Other,Indore,2022-02-02
3,U0004,Aarav Gupta,44,Male,Kolkata,2023-06-02
4,U0005,Sara Sharma,30,Other,Chennai,2024-01-04


In [3]:
# 1.2 Load sales.json
with open('data/sales.json') as f:
    sales_data = json.load(f)

sales_df = pd.DataFrame(sales_data)
print("Sales shape:", sales_df.shape)
sales_df.head()


Sales shape: (1000, 6)


,transaction_id,user_id,product_id,amount,payment_type,date
0,T000001,U0024,P015,67.67,Wallet,2023-02-12
1,T000002,U0196,P044,76.44,UPI,2023-03-24
2,T000003,U0196,P049,104.57,Debit Card,2025-08-21
3,T000004,U0133,P042,102.75,Net Banking,2024-07-23
4,T000005,U0047,P038,23.89,Net Banking,2025-10-04


In [4]:
# 1.3 Load inventory.sql
# the file has CREATE TABLE + INSERT statements, so we just run it inside a
# temporary sqlite database and then read the table back into a dataframe
conn = sqlite3.connect(':memory:')
sql_script = open('data/inventory.sql').read()
conn.executescript(sql_script)

products_df = pd.read_sql_query("SELECT * FROM products", conn)
print("Products shape:", products_df.shape)
products_df.head()


Products shape: (50, 5)


,product_id,product_name,category,price,stock
0,P001,Product_001,Grocery,264.89,371
1,P002,Product_002,Grocery,605.91,150
2,P003,Product_003,Beauty,3027.98,127
3,P004,Product_004,Toys,2600.12,229
4,P005,Product_005,Books,1178.99,18


In [5]:
# 1.4 basic info of all 3 tables
print("----- USERS INFO -----")
print(users_df.info())
print()
print("----- SALES INFO -----")
print(sales_df.info())
print()
print("----- PRODUCTS INFO -----")
print(products_df.info())


----- USERS INFO -----
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   user_id            200 non-null    object
 1   name               200 non-null    object
 2   age                200 non-null    int64 
 3   gender             200 non-null    object
 4   city               200 non-null    object
 5   registration_date  200 non-null    object
dtypes: int64(1), object(5)
memory usage: 9.5+ KB
None

----- SALES INFO -----
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   transaction_id  1000 non-null   object 
 1   user_id         1000 non-null   object 
 2   product_id      1000 non-null   object 
 3   amount          1000 non-null   float64
 4   payment_type    1000 non-null   object

In [6]:
# 1.5 checking data types and missing values
print("Missing values in users:\n", users_df.isnull().sum())
print()
print("Missing values in sales:\n", sales_df.isnull().sum())
print()
print("Missing values in products:\n", products_df.isnull().sum())


Missing values in users:
 user_id              0
name                 0
age                  0
gender               0
city                 0
registration_date    0
dtype: int64

Missing values in sales:
 transaction_id    0
user_id           0
product_id        0
amount            0
payment_type      0
date              0
dtype: int64

Missing values in products:
 product_id      0
product_name    0
category        0
price           0
stock           0
dtype: int64


**Observation:** All three files already look fairly clean at first glance (no missing values shown above).
But we still need to check for hidden problems like wrong date formats, negative prices/amounts,
duplicate rows, etc. before we trust the data. We check that below.

## 2. Data Cleaning

In [7]:
# keep a copy of the original sizes so we can compare "before vs after" at the end
before_cleaning = {
    'users_rows': len(users_df),
    'sales_rows': len(sales_df),
    'products_rows': len(products_df)
}

missing_before = {
    'users': users_df.isnull().sum().sum(),
    'sales': sales_df.isnull().sum().sum(),
    'products': products_df.isnull().sum().sum()
}
print("Rows before cleaning:", before_cleaning)
print("Total missing values before cleaning:", missing_before)


Rows before cleaning: {'users_rows': 200, 'sales_rows': 1000, 'products_rows': 50}
Total missing values before cleaning: {'users': np.int64(0), 'sales': np.int64(0), 'products': np.int64(0)}


In [8]:
# 2.1 check for duplicate rows
print("Duplicate rows in users:", users_df.duplicated().sum())
print("Duplicate rows in sales:", sales_df.duplicated().sum())
print("Duplicate rows in products:", products_df.duplicated().sum())

# drop duplicates if any (won't change anything if there are none)
users_df = users_df.drop_duplicates()
sales_df = sales_df.drop_duplicates()
products_df = products_df.drop_duplicates()


Duplicate rows in users: 0
Duplicate rows in sales: 0
Duplicate rows in products: 0


In [9]:
# 2.2 Handle missing numerical data using SimpleImputer (mean strategy)
from sklearn.impute import SimpleImputer

num_cols_sales = ['amount']
num_imputer = SimpleImputer(strategy='mean')
sales_df[num_cols_sales] = num_imputer.fit_transform(sales_df[num_cols_sales])

num_cols_products = ['price', 'stock']
num_imputer2 = SimpleImputer(strategy='mean')
products_df[num_cols_products] = num_imputer2.fit_transform(products_df[num_cols_products])

num_cols_users = ['age']
num_imputer3 = SimpleImputer(strategy='mean')
users_df[num_cols_users] = num_imputer3.fit_transform(users_df[num_cols_users])

print("Numeric missing values handled (if any existed).")


Numeric missing values handled (if any existed).


In [10]:
# 2.3 Handle missing categorical data using most frequent value
from sklearn.impute import SimpleImputer

cat_cols_users = ['gender', 'city']
cat_imputer = SimpleImputer(strategy='most_frequent')
users_df[cat_cols_users] = cat_imputer.fit_transform(users_df[cat_cols_users])

cat_cols_sales = ['payment_type']
cat_imputer2 = SimpleImputer(strategy='most_frequent')
sales_df[cat_cols_sales] = cat_imputer2.fit_transform(sales_df[cat_cols_sales])

cat_cols_products = ['category']
cat_imputer3 = SimpleImputer(strategy='most_frequent')
products_df[cat_cols_products] = cat_imputer3.fit_transform(products_df[cat_cols_products])

print("Categorical missing values handled (if any existed).")


Categorical missing values handled (if any existed).


In [11]:
# 2.4 KNN Imputer (optional enhancement) - shown here on the users numeric data
# KNN Imputer needs numeric columns, so we just demo it on 'age'
from sklearn.impute import KNNImputer

knn_imputer = KNNImputer(n_neighbors=3)
users_df[['age']] = knn_imputer.fit_transform(users_df[['age']])
print("KNN imputer applied on age column (demo only, age had no missing values here).")


KNN imputer applied on age column (demo only, age had no missing values here).


In [12]:
# 2.5 Fix invalid / inconsistent entries
# a) dates should follow YYYY-MM-DD format -> convert with pandas, invalid ones become NaT
sales_df['date'] = pd.to_datetime(sales_df['date'], errors='coerce', format='%Y-%m-%d')
users_df['registration_date'] = pd.to_datetime(users_df['registration_date'], errors='coerce', format='%Y-%m-%d')

bad_dates_sales = sales_df['date'].isnull().sum()
bad_dates_users = users_df['registration_date'].isnull().sum()
print("Invalid dates found in sales:", bad_dates_sales)
print("Invalid dates found in users:", bad_dates_users)

# b) amount and price should never be negative -> fix by taking absolute value
neg_amount_count = (sales_df['amount'] < 0).sum()
sales_df['amount'] = sales_df['amount'].abs()

neg_price_count = (products_df['price'] < 0).sum()
products_df['price'] = products_df['price'].abs()

print("Negative amounts fixed:", neg_amount_count)
print("Negative prices fixed:", neg_price_count)

# c) age should be a realistic value (say between 10 and 100), anything else is invalid
invalid_age_count = users_df[(users_df['age'] < 10) | (users_df['age'] > 100)].shape[0]
users_df = users_df[(users_df['age'] >= 10) & (users_df['age'] <= 100)]
print("Invalid age rows removed:", invalid_age_count)


Invalid dates found in sales: 0
Invalid dates found in users: 0
Negative amounts fixed: 0
Negative prices fixed: 0
Invalid age rows removed: 0


In [13]:
missing_after_cleaning = {
    'users': users_df.isnull().sum().sum(),
    'sales': sales_df.isnull().sum().sum(),
    'products': products_df.isnull().sum().sum()
}
print("Total missing values after cleaning:", missing_after_cleaning)


Total missing values after cleaning: {'users': np.int64(0), 'sales': np.int64(0), 'products': np.int64(0)}


## 3. Outlier Handling

We check for outliers in `amount` (sales) and `price` (products) using both the **Z-score** method and the **IQR** method, and compare them.

In [14]:
from scipy import stats

# ---- Z-score method on sales amount ----
z_scores = np.abs(stats.zscore(sales_df['amount']))
outliers_z = sales_df[z_scores > 3]
print("Outliers found by Z-score method (amount):", len(outliers_z))

# ---- IQR method on sales amount ----
Q1 = sales_df['amount'].quantile(0.25)
Q3 = sales_df['amount'].quantile(0.75)
IQR = Q3 - Q1
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers_iqr = sales_df[(sales_df['amount'] < lower_limit) | (sales_df['amount'] > upper_limit)]
print("Outliers found by IQR method (amount):", len(outliers_iqr))


Outliers found by Z-score method (amount): 15
Outliers found by IQR method (amount): 53


**Comparison:** The Z-score method assumes the data is roughly normally distributed, while the IQR method
does not make that assumption and is generally safer for skewed data like purchase amounts (a few
customers spend a lot more than others). So for this dataset **IQR is more suitable**, and we use it
to decide which rows to treat.

In [15]:
outlier_count_before = len(outliers_iqr)

# Instead of simply deleting these rows (we don't want to lose real transactions),
# we cap them using Winsorization: any value above the upper limit is pulled down
# to the upper limit, and any value below the lower limit is pushed up to the lower limit.
sales_df['amount'] = np.where(sales_df['amount'] > upper_limit, upper_limit,
                        np.where(sales_df['amount'] < lower_limit, lower_limit, sales_df['amount']))

# recheck how many outliers remain after winsorization
Q1_after = sales_df['amount'].quantile(0.25)
Q3_after = sales_df['amount'].quantile(0.75)
IQR_after = Q3_after - Q1_after
lower_after = Q1_after - 1.5 * IQR_after
upper_after = Q3_after + 1.5 * IQR_after
outlier_count_after = sales_df[(sales_df['amount'] < lower_after) | (sales_df['amount'] > upper_after)].shape[0]

print("Outliers before winsorization:", outlier_count_before)
print("Outliers after winsorization:", outlier_count_after)


Outliers before winsorization: 53
Outliers after winsorization: 0


## 4. Data Transformation

In [16]:
# 4.1 Split date columns into day, month, year
sales_df['purchase_day'] = sales_df['date'].dt.day
sales_df['purchase_month'] = sales_df['date'].dt.month
sales_df['purchase_year'] = sales_df['date'].dt.year

users_df['reg_day'] = users_df['registration_date'].dt.day
users_df['reg_month'] = users_df['registration_date'].dt.month
users_df['reg_year'] = users_df['registration_date'].dt.year

sales_df[['date', 'purchase_day', 'purchase_month', 'purchase_year']].head()


,date,purchase_day,purchase_month,purchase_year
0,2023-02-12,12,2,2023
1,2023-03-24,24,3,2023
2,2025-08-21,21,8,2025
3,2024-07-23,23,7,2024
4,2025-10-04,4,10,2025


In [17]:
# 4.2 Encoding categorical variables

# Label Encoding for a binary-like column -> we will treat 'gender' as label encoded
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
users_df['gender_label'] = le.fit_transform(users_df['gender'])
print(dict(zip(le.classes_, le.transform(le.classes_))))
users_df[['gender', 'gender_label']].head()


{'Female': np.int64(0), 'Male': np.int64(1), 'Other': np.int64(2)}


,gender,gender_label
0,Other,2
1,Other,2
2,Other,2
3,Male,1
4,Other,2


In [18]:
# One-Hot Encoding for nominal columns (payment_type has no natural order)
sales_df = pd.get_dummies(sales_df, columns=['payment_type'], prefix='pay')
sales_df.head()


,transaction_id,user_id,product_id,amount,date,purchase_day,purchase_month,purchase_year,pay_Cash,pay_Credit Card,pay_Debit Card,pay_Net Banking,pay_UPI,pay_Wallet
0,T000001,U0024,P015,67.67,2023-02-12,12,2,2023,False,False,False,False,False,True
1,T000002,U0196,P044,76.44,2023-03-24,24,3,2023,False,False,False,False,True,False
2,T000003,U0196,P049,104.57,2025-08-21,21,8,2025,False,False,True,False,False,False
3,T000004,U0133,P042,102.75,2024-07-23,23,7,2024,False,False,False,True,False,False
4,T000005,U0047,P038,23.89,2025-10-04,4,10,2025,False,False,False,True,False,False


In [19]:
# Ordinal Encoding for ordered categorical variable
# we will create the "spending group" column below (Low/Medium/High) and encode it
# in order, since Low < Medium < High has a real ranking


In [20]:
# 4.3 Binning - segment customers into spending groups based on total amount spent
customer_total_spend = sales_df.groupby('user_id')['amount'].sum()

# 3 equal-width-ish bins using quantiles so groups are balanced
spend_bins = pd.qcut(customer_total_spend, q=3, labels=['Low', 'Medium', 'High'])
spending_group_df = spend_bins.reset_index()
spending_group_df.columns = ['user_id', 'spending_group']

# ordinal encoding of the spending group (Low=0, Medium=1, High=2)
order_map = {'Low': 0, 'Medium': 1, 'High': 2}
spending_group_df['spending_group_encoded'] = spending_group_df['spending_group'].map(order_map)

spending_group_df.head()


,user_id,spending_group,spending_group_encoded
0,U0001,Low,0
1,U0002,High,2
2,U0003,Low,0
3,U0004,Medium,1
4,U0005,Medium,1


In [21]:
# 4.4 Log and square root transformation to reduce skewness of 'amount'
sales_df['amount_log'] = np.log1p(sales_df['amount'])       # log1p handles 0 values safely
sales_df['amount_sqrt'] = np.sqrt(sales_df['amount'])

print("Skew before transform:", sales_df['amount'].skew())
print("Skew after log transform:", sales_df['amount_log'].skew())
print("Skew after sqrt transform:", sales_df['amount_sqrt'].skew())


Skew before transform: 0.9101086040815545
Skew after log transform: -0.1772341929471458
Skew after sqrt transform: 0.40605371526585177


## 5. Feature Scaling

In [22]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

scale_cols = ['amount']

standard_scaler = StandardScaler()
sales_df['amount_standard_scaled'] = standard_scaler.fit_transform(sales_df[scale_cols])

minmax_scaler = MinMaxScaler()
sales_df['amount_minmax_scaled'] = minmax_scaler.fit_transform(sales_df[scale_cols])

sales_df[['amount', 'amount_standard_scaled', 'amount_minmax_scaled']].describe()


,amount,amount_standard_scaled,amount_minmax_scaled
count,1000.00,1000.00,1000.00
mean,65.03,-0.00,0.40
std,36.13,1.00,0.25
min,7.81,-1.58,0.00
25%,37.74,-0.76,0.21
50%,56.39,-0.24,0.34
75%,82.94,0.50,0.53
max,150.72,2.37,1.00


**Comparison:** StandardScaler centers the data around mean 0 with standard deviation 1, so it still
has negative values. MinMaxScaler squashes everything between 0 and 1. MinMaxScaler is easier to
interpret here, but StandardScaler is usually better if we later use models that assume normal-ish
data (like linear regression or KNN).

## 6. Feature Construction

In [23]:
# merge sales with products so we know the category of every transaction
sales_full = sales_df.merge(products_df[['product_id', 'category', 'price']], on='product_id', how='left')

# 6.1 Average monthly spend per customer
sales_full['year_month'] = sales_full['date'].dt.to_period('M')
monthly_spend = sales_full.groupby(['user_id', 'year_month'])['amount'].sum().reset_index()
avg_monthly_spend = monthly_spend.groupby('user_id')['amount'].mean().reset_index()
avg_monthly_spend.columns = ['user_id', 'avg_monthly_spend']

# 6.2 Frequency of purchase (number of transactions per customer)
purchase_frequency = sales_full.groupby('user_id')['transaction_id'].count().reset_index()
purchase_frequency.columns = ['user_id', 'purchase_frequency']

# 6.3 Days since last purchase (using the most recent date in the dataset as "today")
reference_date = sales_full['date'].max()
last_purchase = sales_full.groupby('user_id')['date'].max().reset_index()
last_purchase['days_since_last_purchase'] = (reference_date - last_purchase['date']).dt.days
last_purchase = last_purchase[['user_id', 'days_since_last_purchase']]

# 6.4 Category-wise total expenditure per customer
category_expenditure = sales_full.pivot_table(index='user_id', columns='category',
                                               values='amount', aggfunc='sum', fill_value=0)
category_expenditure.columns = ['spend_' + str(c).lower() for c in category_expenditure.columns]
category_expenditure = category_expenditure.reset_index()

print("New feature tables created:")
print("avg_monthly_spend:", avg_monthly_spend.shape)
print("purchase_frequency:", purchase_frequency.shape)
print("last_purchase:", last_purchase.shape)
print("category_expenditure:", category_expenditure.shape)


New feature tables created:
avg_monthly_spend: (200, 2)
purchase_frequency: (200, 2)
last_purchase: (200, 2)
category_expenditure: (200, 9)


## 7. Final Dataset Preparation

In [24]:
# merge everything into one customer-level table
final_df = users_df.copy()
final_df = final_df.merge(avg_monthly_spend, on='user_id', how='left')
final_df = final_df.merge(purchase_frequency, on='user_id', how='left')
final_df = final_df.merge(last_purchase, on='user_id', how='left')
final_df = final_df.merge(category_expenditure, on='user_id', how='left')
final_df = final_df.merge(spending_group_df, on='user_id', how='left')

# customers with no purchases at all -> fill their new numeric features with 0
new_feature_cols = ['avg_monthly_spend', 'purchase_frequency', 'days_since_last_purchase'] \
                    + [c for c in category_expenditure.columns if c != 'user_id']
final_df[new_feature_cols] = final_df[new_feature_cols].fillna(0)

print("Final dataset shape:", final_df.shape)
final_df.head()


Final dataset shape: (200, 23)


,user_id,name,age,gender,city,registration_date,reg_day,reg_month,reg_year,gender_label,...,spend_beauty,spend_books,spend_clothing,spend_electronics,spend_grocery,spend_home,spend_sports,spend_toys,spending_group,spending_group_encoded
0,U0001,Vihaan Sharma,35.00,Other,Jaipur,2022-09-08,8,9,2022,2,...,42.43,0.00,0.00,44.93,145.06,0.00,0.00,0.00,Low,0
1,U0002,Sai Reddy,30.00,Other,Hyderabad,2023-11-24,24,11,2023,2,...,187.02,0.00,45.02,0.00,0.00,0.00,0.00,157.00,High,2
2,U0003,Aarohi Gupta,37.00,Other,Indore,2022-02-02,2,2,2022,2,...,44.45,34.70,0.00,0.00,24.37,58.87,0.00,0.00,Low,0
3,U0004,Aarav Gupta,44.00,Male,Kolkata,2023-06-02,2,6,2023,1,...,207.65,0.00,0.00,0.00,0.00,0.00,0.00,78.14,Medium,1
4,U0005,Sara Sharma,30.00,Other,Chennai,2024-01-04,4,1,2024,2,...,103.03,68.07,48.34,0.00,0.00,128.25,0.00,0.00,Medium,1


In [25]:
# 7.1 Final report

records_before = before_cleaning['users_rows']
records_after = len(final_df)

original_feature_count = users_df.shape[1]  # includes engineered date parts already, but ok as base
features_created = final_df.shape[1] - original_feature_count

print("======== FINAL REPORT ========")
print("Records before cleaning (users):", records_before)
print("Records after cleaning + merging:", records_after)
print()
print("Number of new features created (approx):", features_created)
print()
print("Missing values before cleaning:", missing_before)
print("Missing values after cleaning:", missing_after_cleaning)
print()
print("Outliers (sales amount) before treatment:", outlier_count_before)
print("Outliers (sales amount) after treatment:", outlier_count_after)


======== FINAL REPORT ========
Records before cleaning (users): 200
Records after cleaning + merging: 200

Number of new features created (approx): 13

Missing values before cleaning: {'users': np.int64(0), 'sales': np.int64(0), 'products': np.int64(0)}
Missing values after cleaning: {'users': np.int64(0), 'sales': np.int64(0), 'products': np.int64(0)}

Outliers (sales amount) before treatment: 53
Outliers (sales amount) after treatment: 0


## 8. Bonus (Optional)

Instead of installing the full `ydata-profiling` library (which is heavy and needs a lot of extra
packages), a quick manual EDA summary is generated below using plain pandas. This gives the same kind
of information (shape, dtypes, missing values, basic stats) in a lightweight way. If a full HTML
profiling report is needed, it can be generated with:

```python
from ydata_profiling import ProfileReport
ProfileReport(final_df).to_file("eda_report.html")
```

In [26]:
print("Shape:", final_df.shape)
print()
print("Data types:\n", final_df.dtypes)
print()
print("Missing values:\n", final_df.isnull().sum())
print()
print("Summary statistics:")
final_df.describe()


Shape: (200, 23)

Data types:
 user_id                             object
name                                object
age                                float64
gender                              object
city                                object
registration_date           datetime64[ns]
reg_day                              int32
reg_month                            int32
reg_year                             int32
gender_label                         int64
avg_monthly_spend                  float64
purchase_frequency                   int64
days_since_last_purchase             int64
spend_beauty                       float64
spend_books                        float64
spend_clothing                     float64
spend_electronics                  float64
spend_grocery                      float64
spend_home                         float64
spend_sports                       float64
spend_toys                         float64
spending_group                    category
spending_group_encoded 

,age,registration_date,reg_day,reg_month,reg_year,gender_label,avg_monthly_spend,purchase_frequency,days_since_last_purchase,spend_beauty,spend_books,spend_clothing,spend_electronics,spend_grocery,spend_home,spend_sports,spend_toys
count,200.00,200,200.00,200.00,200.00,200.00,200.00,200.00,200.00,200.00,200.00,200.00,200.00,200.00,200.00,200.00,200.00
mean,31.26,2023-04-21 09:50:24,15.93,6.08,2022.84,0.99,70.35,5.00,180.91,38.13,61.03,42.79,27.62,39.72,63.62,22.49,29.77
min,18.00,2022-01-04 00:00:00,1.00,1.00,2022.00,0.00,28.23,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,26.00,2022-07-27 00:00:00,10.00,3.00,2022.00,0.00,57.70,3.00,46.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
50%,31.50,2023-04-08 12:00:00,15.00,6.00,2023.00,1.00,68.49,5.00,132.50,0.00,36.19,0.00,0.00,0.00,45.53,0.00,0.00
75%,35.25,2024-01-05 00:00:00,23.25,9.00,2024.00,2.00,83.73,6.00,249.75,63.20,104.33,71.55,49.79,67.31,102.36,36.01,45.32
max,53.00,2024-09-27 00:00:00,31.00,12.00,2024.00,2.00,155.01,13.00,720.00,275.66,340.41,390.61,190.78,247.27,290.87,259.77,272.09
std,7.27,NaN,8.59,3.24,0.80,0.81,20.32,2.18,166.08,57.18,72.55,61.67,42.99,55.31,67.15,42.78,51.78


In [27]:
# save the final cleaned & engineered dataset
final_df.to_csv('final_cleaned_dataset.csv', index=False)
print("Saved final_cleaned_dataset.csv with shape:", final_df.shape)


Saved final_cleaned_dataset.csv with shape: (200, 23)
